These commands make sure that matplotlib is configured correctly to run in 
docker. It makes matplotlib use a non-graphical backend, but allows you to 
still show plots inline in a notebook. This bypasses docker graphical issues
that usually pop up, and will be needed in most notebooks.

In [12]:
import matplotlib
matplotlib.use('Agg')
%matplotlib inline
import matplotlib.pyplot as plt

# Sodetlib Config Overview

In [4]:
from sodetlib.det_config import DetConfig
from pprint import pprint

## Guide to configurations files

In the current system deployed for SO, there are multiple configuration files that need to be present. These are typically stored in a directory defined by the `$SMURF_CONFIG_DIR` environment variable on the server node.

The config files are:
- `sys_config.yml`: high-level description of the system, including populated slots, IP addresses, and docker tags to use for the software. Also includes paths to subsequent configs. Used by `sodetlib` and `jackhammer` to start and configure the SMuRF system.
- Device config: used by `sodetlib` to store information about the state of the system, per slot. This includes amplifier biases, tuning data, active bands, etc. `sodetlib` operations will modify this file to store the latest state (usually accepting and `update_config` boolean argument).
- `pysmurf` config: used by `pysmurf` (lower-level library) to initialise the hardware with default register values. Specific to an individual system and shouldn't need to be edited once deployed.

Typically a user will interact with the `sys_config.yml` file to configure a system, and the device configs via `sodetlib` operations.

More details can be found in the [`sodetlib` docs](https://sodetlib.readthedocs.io/en/latest/configs.html). Examples from the [SO deployment configs](https://github.com/simonsobs/ocs-deployment-configs/tree/main/lat/smurf-so8-lat) are also available.

In `python`, the `DetConfig` object is used to load system, device, and pysmurf configurations. By default, it will first load the system configuration from `$SMURF_CONFIG_DIR/sys_config.yml`. This config file should contain the device and pysmurf config files to use for each smurf slot, which the `DetConfig` system will then load.

Assuming the default location for `sys_config.yml`, only the slot we wish to access needs to be specified.

## Loading the configuration for a given slot

In [5]:
cfg = DetConfig()
cfg.load_config_files(slot=2)

The system config and device config, can be accessed at `cfg.sys` and `cfg.dev` respectively.

In [6]:
print("Sys config\n" + 13*"-")
pprint(cfg.sys)

Sys config
-------------
{'comm_type': 'eth',
 'crate_id': 1,
 'docker_env': {'ATCA_MONITOR_TAG': 'R1.0.0',
                'CB_HOST': '192.168.0.120',
                'PYSMURF_CLIENT_TAG': 'v4.1.0',
                'SOCS_TAG': 'v0.1.0-3-gaf8873a-dev',
                'SODETLIB_TAG': 'v0.0.1',
                'STREAMER_TAG': 'v0.0.2-v2.1.0-stable'},
 'max_fan_level': 10,
 'meta_register_file': '$OCS_CONFIG_DIR/meta_registers.yaml',
 'shelf_manager': 'shm-smrf-sp01',
 'slot_order': [2],
 'slots': {'SLOT[2]': {'device_config': '$OCS_CONFIG_DIR/device_configs/dev_cfg_demo_test_s2.yaml',
                       'pysmurf_config': '/config/pysmurf_config/experiment_ucsd_k2so_cc02-06_lbOnlyBay0.cfg',
                       'stream_port': 4532}},
 'startup': {'configure_pysmurf': True,
             'reboot': True,
             'run_half_band_test': False,
             'set_crate_fans_to_full': True,
             'start_atca_monitor': False,
             'write_config': False}}


Here we see that the system is configured for one slot 2 in crate 1, and can read off the docker tags that will be used to start up the software and firmware.

The device config is split into experiment config, or `exp`, which contains general config info about the device,
`bands` which contains info about the 8 bands on the slot, and `bias_groups` which contains config info about the 12 bias groups.

In [7]:
print("Exp\n"+5*"-")
pprint(cfg.dev.exp)
print("\nBand[0]\n" + 10*'-')
pprint(cfg.dev.bands[3])
print("\nBiasGroup[0]\n" + 10*'-')
pprint(cfg.dev.bias_groups[0])

Exp
-----
{'amp_50k_Id': 14,
 'amp_50k_Vg': -0.5179895424000001,
 'amp_hemt_Id': 8,
 'amp_hemt_Vg': -0.8299942848,
 'tunefile': '/data/smurf_data/tune/1606763812_tune.npy'}

Band[0]
----------
{'active_subbands': [12,
                     13,
                     14,
                     15,
                     16,
                     17,
                     18,
                     19,
                     20,
                     30,
                     31,
                     32,
                     33,
                     34,
                     35,
                     36,
                     37,
                     38,
                     39,
                     40,
                     41,
                     42,
                     43],
 'dc_att': 0,
 'detectors': [],
 'drive': 12,
 'feedback_end_frac': 0.98,
 'feedback_start_frac': 0.02,
 'flux_ramp_rate_khz': 4,
 'frac_pp': 0.2797240558589269,
 'lms_freq_hz': 20000.0,
 'lms_gain': 5,
 'nphi0': 4,
 'optimized_dri

## Creating pysmurf instance

To create a pysmurf instance based on this configuration setup, simply run: 

In [8]:
S = cfg.get_smurf_control(dump_configs=True)

This will create a control object with the correct epics root, pysmurf config file, etc. and dump all of the config files to S.data_directory. Using this will also correctly set the pysmurf publisher ID correctly based on the crate id and slot number.